In [2]:
from functools import partial

from transformers import CLIPVisionModel 
import torch
from torch import nn
from torchvision import transforms
from PIL import Image

import torch
import torch.nn as nn
from transformers import CLIPVisionModel
from torchvision import transforms

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')



In [5]:
import torch
import os

# Define paths relative to current directory
train_git_path = 'ViT-L-14_features_GIT_train.pt'
test_git_path = 'ViT-L-14_features_GIT_test.pt'
train_vit_path = 'ViT-H-14_features_train.pt'
test_vit_path = 'ViT-H-14_features_test.pt'

# Load features with pickle protocol
train_pixel_img_feature = torch.load(train_git_path, map_location='cpu', pickle_module=torch.serialization.pickle)
test_pixel_img_feature = torch.load(test_git_path, map_location='cpu', pickle_module=torch.serialization.pickle)
train_img_feature = torch.load(train_vit_path, map_location='cpu', pickle_module=torch.serialization.pickle)
test_img_feature = torch.load(test_vit_path, map_location='cpu', pickle_module=torch.serialization.pickle)

# If they are dictionaries, extract 'img_features'
if isinstance(train_pixel_img_feature, dict):
    train_pixel_img_feature = train_pixel_img_feature['img_features']
if isinstance(test_pixel_img_feature, dict):
    test_pixel_img_feature = test_pixel_img_feature['img_features']
if isinstance(train_img_feature, dict):
    train_img_feature = train_img_feature['img_features']
if isinstance(test_img_feature, dict):
    test_img_feature = test_img_feature['img_features']

# Add dimension to ViT features if needed
train_img_feature = train_img_feature.unsqueeze(1)
test_img_feature = test_img_feature.unsqueeze(1)

print("GIT train shape:", train_pixel_img_feature.shape)
print("GIT test shape:", test_pixel_img_feature.shape)
print("ViT-H-14 train shape:", train_img_feature.shape)

RuntimeError: PytorchStreamReader failed reading zip archive: not a ZIP archive

In [1]:
import torch

# Load GIT features (shape: [16540, 257, 1024])
train_pixel_img_feature = torch.load(r'D:\Universität\Master\4. Semester\Forschungspraxis\fp_python\EEG_Image_decode-main\Generation\ViT-L-14_features_GIT_train.pt')['img_features']
test_pixel_img_feature = torch.load(r'D:\Universität\Master\4. Semester\Forschungspraxis\fp_python\EEG_Image_decode-main\Generation\ViT-L-14_features_GIT_test.pt')['img_features']

# Load ViT-H-14 features (shape: [16540, 1024])
train_img_feature = torch.load(r'D:\Universität\Master\4. Semester\Forschungspraxis\fp_python\EEG_Image_decode-main\Generation\ViT-H-14_features_train.pt')['img_features'].unsqueeze(1)
test_img_feature = torch.load(r'D:\Universität\Master\4. Semester\Forschungspraxis\fp_python\EEG_Image_decode-main\Generation\ViT-H-14_features_test.pt')['img_features'].unsqueeze(1)

print("GIT train shape:", train_pixel_img_feature.shape)
print("GIT test shape:", test_pixel_img_feature.shape)
print("ViT-H-14 train shape:", train_img_feature.shape)
print("ViT-H-14 test shape:", test_img_feature.shape)

RuntimeError: PytorchStreamReader failed reading zip archive: not a ZIP archive

In [ ]:
import torch
import os

def safe_load_tensor(file_path):
    try:
        # First try loading as a regular tensor
        data = torch.load(file_path, map_location='cpu')
        if isinstance(data, dict) and 'img_features' in data:
            return data['img_features']
        return data
    except Exception as e:
        print(f"Error loading {file_path}: {str(e)}")
        return None

# Define paths
base_path = r'D:\Universität\Master\4. Semester\Forschungspraxis\fp_python\EEG_Image_decode-main\Generation'
train_git_path = os.path.join(base_path, 'ViT-L-14_features_GIT_train.pt')
test_git_path = os.path.join(base_path, 'ViT-L-14_features_GIT_test.pt')
train_vit_path = os.path.join(base_path, 'ViT-H-14_features_train.pt')
test_vit_path = os.path.join(base_path, 'ViT-H-14_features_test.pt')

# Load features with error handling
train_pixel_img_feature = safe_load_tensor(train_git_path)
test_pixel_img_feature = safe_load_tensor(test_git_path)
train_img_feature = safe_load_tensor(train_vit_path)
test_img_feature = safe_load_tensor(test_vit_path)

# Print shapes if loading was successful
if train_pixel_img_feature is not None:
    print("GIT train shape:", train_pixel_img_feature.shape)
if test_pixel_img_feature is not None:
    print("GIT test shape:", test_pixel_img_feature.shape)
if train_img_feature is not None:
    print("ViT-H-14 train shape:", train_img_feature.shape)
if test_img_feature is not None:
    print("ViT-H-14 test shape:", test_img_feature.shape)

Error loading D:\Universität\Master\4. Semester\Forschungspraxis\fp_python\EEG_Image_decode-main\Generation\ViT-L-14_features_GIT_train.pt: PytorchStreamReader failed reading zip archive: not a ZIP archive
GIT test shape: torch.Size([200, 257, 1024])
ViT-H-14 train shape: torch.Size([16540, 1024])
ViT-H-14 test shape: torch.Size([200, 1024])


RuntimeError: PytorchStreamReader failed reading zip archive: not a ZIP archive

In [ ]:
test_img_feature.shape

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from einops.layers.torch import Rearrange, Reduce

# Define the neural network
class PixelProjector(nn.Sequential):
    def __init__(self, proj_dim=1024):
        super().__init__(
            Rearrange('B C L->B L C'),    
            nn.Linear(1, 257),
            nn.LayerNorm(257),
            Rearrange('B L C->B C L'),
            nn.Linear(1024, 1024),
            nn.LayerNorm(proj_dim),
            )
        
        

# Instantiate the model, loss function, and optimizer

model = PixelProjector(proj_dim=1024).to(torch.bfloat16).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

# Prepare data loaders
train_dataset = TensorDataset(train_img_feature, train_pixel_img_feature)
test_dataset = TensorDataset(test_img_feature, test_pixel_img_feature)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Training loop
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(torch.bfloat16).to(device), targets.to(torch.bfloat16).to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")

# Testing loop
model.eval()
test_loss = 0.0
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(torch.bfloat16).to(device), targets.to(torch.bfloat16).to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        test_loss += loss.item()

print(f"Test Loss: {test_loss/len(test_loader)}")

# Save the trained model
torch.save(model.state_dict(), '/root/autodl-tmp/Workspace/EEG_caption/model_weights/PixelProjector_best.bin')
print("Model saved as PixelProjector.bin")


In [8]:
# Save the trained model
torch.save(model.state_dict(), '/root/autodl-tmp/Workspace/EEG_caption/model_weights/PixelProjector_best.bin')
print("Model saved as PixelProjector.bin")

Model saved as PixelProjector.bin
